# Laboratorio 4: asistente de compras por voz

Se implementa un asistente que recibe consultas habladas en español, recupera productos de un catálogo derivado de Amazon ESCI, ejecuta operaciones sobre una tienda simulada y responde con una voz clonada.

**Flujo por turno:** grabación → ASR (Whisper Turbo) → agente (Qwen3-4B con herramientas) → recuperación híbrida (BM25 + E5) y tienda simulada → TTS con clonación (Qwen3-TTS).

| Componente | Modelo | Justificación |
|---|---|---|
| ASR | `faster-whisper` Turbo, CPU INT8 | Buena precisión en español y no ocupa memoria de la GPU. |
| Embeddings | `intfloat/multilingual-e5-small` | Multilingüe, liviano (118 M parámetros) y rápido para 9 554 productos. |
| LLM | `Qwen/Qwen3-4B-Instruct-2507`, 4 bits NF4 | Soporta llamadas a herramientas y cabe en una T4 junto con el TTS (≈3 GB). |
| TTS | `Qwen/Qwen3-TTS-12Hz-0.6B-Base`, FP32 | Clonación de voz a partir de una grabación corta; FP32 evita BF16, que la T4 no soporta. |
| Modelo base de búsqueda | BM25 (`bm25s`, stemming en español) | Búsqueda léxica exigida como referencia. |

No se entrena ningún modelo. El diseño de la recuperación, el control del agente, la memoria, las herramientas y las validaciones se implementan en los scripts de esta carpeta.

## Instrucciones de ejecución

1. Abrir el cuaderno en Google Colab con GPU (Entorno de ejecución → Cambiar tipo de entorno → GPU T4 o superior).
2. Ejecutar las celdas en orden. La primera clona el repositorio e instala las versiones utilizadas.
3. Las grabaciones de voz no se publican en el repositorio. Se suben como `audios.zip` en la celda correspondiente (contiene `referencia.wav` y `E0X_tY.wav`).

| Archivo | Contenido |
|---|---|
| `datos.py` | Descarga de ESCI y preparación del catálogo y de las consultas. |
| `tienda.py` | Tienda simulada: precios, existencias, carrito, cotizaciones y pedidos con semilla fija. |
| `recuperacion.py` | Índices BM25 y E5, y fusión híbrida por rangos recíprocos (RRF). |
| `agente.py` | Bucle del agente, herramientas, memoria y validaciones. |
| `escenarios.py` | 20 escenarios de evaluación y 4 de desarrollo con su verificación automática. |
| `voz.py` | ASR, TTS con clonación y métricas de voz. |
| `evaluar_recuperacion.py`, `evaluar_escenarios.py`, `evaluar_voz.py` | Experimentos. |

Versiones: `qwen-tts==0.1.1`, `transformers==4.57.3`, `faster-whisper==1.2.1`, `bitsandbytes==0.48.2`, `sentence-transformers==5.1.2`, `datasets==4.4.1`, `av<16` (las versiones nuevas de PyAV no son compatibles con `faster-whisper` 1.2.1). Semilla: 42.

In [ ]:
!apt-get -qq update && apt-get -qq install -y ffmpeg sox
%cd /content
![ -d UTEC-deep_learning ] || git clone -q https://github.com/FernandoIGD/UTEC-deep_learning.git
%cd /content/UTEC-deep_learning/lab04
!git pull -q
!pip -q install "qwen-tts==0.1.1" "transformers==4.57.3" "faster-whisper==1.2.1" "bitsandbytes==0.48.2" "sentence-transformers==5.1.2" "datasets==4.4.1" "av<16" bm25s PyStemmer jiwer

In [ ]:
import json, time
import pandas as pd
import torch
from IPython.display import Audio, display

DISPOSITIVO = "cuda"
print(torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 1. Datos

Se usa la versión reducida (`small_version = 1`) en español. Se eligen con semilla fija 100 consultas de `train` (desarrollo) y 200 de `test` (evaluación), y se incluyen en el catálogo **todos** sus productos anotados. Se agregan 1 500 productos en español sin anotaciones para que el catálogo no contenga solo candidatos. Las etiquetas ESCI se usan únicamente para evaluar; el agente no puede consultarlas.

In [ ]:
from datos import preparar
catalogo, consultas = preparar()
print(f"Catálogo: {len(catalogo)} productos")
print("Consultas por split:", consultas.groupby("split")["query_id"].nunique().to_dict())
print("Etiquetas:", consultas["esci_label"].value_counts().to_dict())
catalogo[["product_id", "product_title", "product_brand", "product_color"]].head()

## 2. Tienda simulada

Los precios (distribución lognormal) y las existencias (1 a 30 unidades, 8 % agotados) se generan con semilla 42. `restaurar()` regenera exactamente el mismo estado. Todas las validaciones de cantidades, existencias y presupuesto se hacen en código:

- `fijar_cantidad` rechaza cantidades no enteras, negativas o superiores a las existencias.
- `cotizar` guarda los precios del momento y marca los problemas (existencias o presupuesto).
- `confirmar` vuelve a validar todo; si algo cambió, no modifica nada. Si todo es válido, descuenta las existencias y registra el pedido en un solo paso. Confirmar dos veces la misma cotización devuelve el pedido existente.
- El agente solo puede confirmar una cotización presentada en un turno anterior.

In [ ]:
from tienda import Tienda
tienda = Tienda(catalogo["product_id"])
ejemplo = next(p for p in tienda.ids if tienda.existencias[p] >= 2)
print(tienda.disponibilidad(ejemplo))
tienda.fijar_cantidad(ejemplo, 2); c = tienda.cotizar(presupuesto=1000)
print(tienda.confirmar(c["cotizacion_id"], 1000)["pedido"]["pedido_id"], tienda.confirmar(c["cotizacion_id"], 1000)["repetido"])
tienda.restaurar(); print("Restaurada:", tienda.disponibilidad(ejemplo), tienda.pedidos)

## 3. Recuperación: modelo base y ablación

nDCG@10 con ganancias E = 1, S = 0,1, C = 0,01, I = 0. Se reportan dos variantes: ordenar **solo los candidatos anotados** de cada consulta (lo solicitado) y ordenar **todo el catálogo** (lo que hace el agente; los productos no anotados reciben ganancia 0). La constante `k` de RRF se elige con las consultas de desarrollo; las de evaluación no se usan para ajustar.

- **Modelo base:** BM25.
- **Sistema:** fusión híbrida BM25 + E5.
- **Ablaciones:** sin BM25 (solo E5) y con solo el título del producto.

In [ ]:
!python evaluar_recuperacion.py cuda
recuperacion = json.load(open("resultados/recuperacion.json"))
pd.DataFrame(recuperacion["prueba"])

In [ ]:
import matplotlib.pyplot as plt
tabla = pd.DataFrame(recuperacion["prueba"]).set_index("sistema")
tabla.plot.barh(figsize=(8, 3.5), xlim=(0, 1), title="nDCG@10 en 200 consultas de evaluación")
plt.tight_layout(); plt.show()

## 4. Carga de modelos y memoria

Whisper se ejecuta en CPU con INT8; el LLM (4 bits) y el TTS (FP32) comparten la GPU. Se mide por separado la carga inicial de cada modelo.

In [ ]:
from evaluar_voz import cargar_modelos
agente, asr, tts, carga = cargar_modelos(DISPOSITIVO)
print("Carga inicial (s):", {k: round(v, 1) for k, v in carga.items()})
print(f"Memoria GPU ocupada: {torch.cuda.memory_allocated() / 1e9:.2f} GB | máximo: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")

## 5. Escenarios operativos por texto

Se ejecutan los 20 escenarios de evaluación con su estado inicial reproducible (`restaurar()` y una preparación propia, por ejemplo agotar productos). Cada escenario tiene criterios verificados por código. Además, se cuentan los incumplimientos de restricciones: pedido sobre el presupuesto, existencias negativas, pedidos duplicados y pedidos registrados sin la aceptación del usuario. El *prompt* del agente se ajustó solo con los 4 escenarios de desarrollo.

In [ ]:
from escenarios import EVALUACION, ejecutar, guardar
from evaluar_escenarios import resumir
registros_texto = []
for escenario in EVALUACION:
    r = ejecutar(escenario, agente); guardar(r, "escenarios_evaluacion"); registros_texto.append(r)
tabla_texto = resumir(registros_texto)
print(f"Completados: {tabla_texto['completado'].mean():.0%} | incumplimientos: {sum(len(r['violaciones']) for r in registros_texto)}")
tabla_texto

In [ ]:
def mostrar(r):
    for i, (u, a) in enumerate(zip(r["entradas"], r["respuestas"]), 1):
        print(f"[{i}] Usuario: {u}\n    Agente: {a}")
        for o in r["operaciones"]:
            if o["turno"] == i:
                print("      ->", o["herramienta"], json.dumps(o["argumentos"], ensure_ascii=False)[:120])
    print("Pedidos:", [(p["pedido_id"], p["total"]) for p in r["estado_final"]["pedidos"]], "| completado:", r["completado"])

mostrar(registros_texto[5])

## 6. Escenarios por voz

Seis escenarios se prueban con audio (tres o más con varias intervenciones del usuario). Cada turno grabado se transcribe con Whisper, se procesa con el agente y la respuesta se sintetiza con la voz clonada. Se conservan los audios, las transcripciones, las operaciones y el estado final en `resultados/`.

En E05, el primer turno se lee con «catorce» en lugar de «cuatro» para reproducir de forma controlada un error de transcripción; el turno siguiente lo corrige.

In [ ]:
from google.colab import files
subido = files.upload()  # archivo .zip con referencia.wav y E0X_tY.wav
nombre = next(iter(subido))
!unzip -o -q -j "{nombre}" "*.wav" -d audios && ls audios | head -30

In [ ]:
from evaluar_voz import ejecutar_voz, resumen_voz
from voz import clonar_voz
inicio = time.perf_counter()
voz = clonar_voz(tts, "audios/referencia.wav")
print(f"Preparación de la voz clonada: {time.perf_counter() - inicio:.1f} s")
registros_voz = []
for escenario in [e for e in EVALUACION if e.get("voz")]:
    r = ejecutar_voz(escenario, agente, asr, tts, voz); guardar(r, "escenarios_voz"); registros_voz.append(r)
    print(r["id"], "completado" if r["completado"] else "fallido", r["violaciones"])
tabla_voz = resumen_voz(registros_voz)
tabla_voz

In [ ]:
for v in registros_voz[0]["voz"]:
    print(f"Dicho:     {v['dicho']}\nTranscrito: {v['transcrito']}\n")
display(Audio("audios/referencia.wav"))
display(Audio("resultados/audios/E01_t2_respuesta.wav"))

### 6.1 Errores del ASR y comparación con la transcripción correcta

Para aislar los errores del ASR, los mismos seis escenarios se ejecutan con la transcripción correcta como entrada (sección 5). Se reportan el WER y los errores en cantidades, importes y productos.

In [ ]:
comparacion = pd.DataFrame({
    "id": [r["id"] for r in registros_voz],
    "completado_voz": [r["completado"] for r in registros_voz],
    "completado_texto": [next(t["completado"] for t in registros_texto if t["id"] == r["id"]) for r in registros_voz],
})
errores = tabla_voz.groupby("id")[["wer", "errores_cantidades", "errores_importes", "errores_productos"]].agg({"wer": "mean", "errores_cantidades": "sum", "errores_importes": "sum", "errores_productos": "sum"})
comparacion.merge(errores, on="id").round(3)

### 6.2 Inteligibilidad y semejanza de la voz sintetizada

- **Inteligibilidad:** WER entre el texto enviado al TTS y su transcripción con Whisper (menor es mejor).
- **Semejanza:** similitud coseno entre el embedding de hablante (x-vector) de la grabación de referencia y el de cada respuesta. Se usa el codificador de hablante de Qwen3-TTS, por lo que la medida puede favorecer al propio modelo.

In [ ]:
print(f"WER del TTS: {tabla_voz['wer_tts'].mean():.3f} | similitud media: {tabla_voz['similitud'].mean():.3f} (mín. {tabla_voz['similitud'].min():.3f})")

### 6.3 Latencia por etapa

Tiempo desde que termina la grabación hasta que la respuesta puede reproducirse: ASR + LLM + herramientas + TTS. La carga inicial de los modelos se reporta aparte (sección 4).

In [ ]:
etapas = tabla_voz[["s_asr", "s_llm", "s_herramientas", "s_tts", "s_total"]]
display(etapas.describe().loc[["mean", "50%", "max"]].round(2))
etapas[["s_asr", "s_llm", "s_herramientas", "s_tts"]].mean().plot.bar(title="Tiempo medio por etapa (s)", figsize=(6, 3)); plt.tight_layout(); plt.show()

## 7. Demostración interactiva

Grabación desde el micrófono del navegador con el componente proporcionado. Cada ejecución procesa un turno de la conversación actual y muestra el tiempo real de cada etapa.

In [ ]:
from IPython.display import Javascript
from google.colab import output
import base64, subprocess

def record(out_wav="input.wav", sr=24000):
    display(Javascript('''
    async function recordAudio() {
      const stream = await navigator.mediaDevices.getUserMedia({audio: true});
      const box = document.createElement('div');
      box.innerHTML = '<button>Grabar</button> <button disabled>Detener</button>';
      document.body.appendChild(box);
      const [start, stop] = box.querySelectorAll('button');
      const recorder = new MediaRecorder(stream);
      const chunks = [];
      recorder.ondataavailable = event => chunks.push(event.data);
      return await new Promise(resolve => {
        recorder.onstop = () => {
          stream.getTracks().forEach(track => track.stop());
          const reader = new FileReader();
          reader.onloadend = () => { box.remove(); resolve(reader.result.split(',')[1]); };
          reader.readAsDataURL(new Blob(chunks, {type: recorder.mimeType}));
        };
        start.onclick = () => { recorder.start(); start.disabled = true; stop.disabled = false; };
        stop.onclick = () => { stop.disabled = true; recorder.stop(); };
      });
    }
    '''))
    data = base64.b64decode(output.eval_js("recordAudio()"))
    open("input.webm", "wb").write(data)
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", "input.webm", "-ac", "1", "-ar", str(sr), out_wav], check=True)
    return out_wav

agente.tienda.restaurar(); agente.reiniciar()

In [ ]:
from voz import transcribir, sintetizar
import soundfile as sf
audio = record()
texto, s_asr = transcribir(asr, audio); print("ASR:", texto)
respuesta, s_llm, s_herr = agente.responder(texto); print("Agente:", respuesta)
onda, sr, s_tts = sintetizar(tts, voz, respuesta); sf.write("respuesta.wav", onda, sr)
print(f"ASR {s_asr:.1f} s | LLM {s_llm:.1f} s | herramientas {s_herr:.2f} s | TTS {s_tts:.1f} s | total {s_asr + s_llm + s_herr + s_tts:.1f} s")
display(Audio("respuesta.wav", autoplay=True))
print("Carrito:", agente.ver_carrito())

## 8. Análisis de resultados

Se completa después de la ejecución (aciertos, fallos, causas y limitaciones).